# Chapter 3: Introduction to Text Generation
**Module 03 - Deep Learning for Text with PyTorch**  
*Source integrated from `chapter3.pdf`*

This chapter moves from classifying text to generating it. It covers character-level RNNs, GANs for synthetic sequence-like data, pre-trained generation models, translation with T5, and generation metrics.


## Learning Objectives

By the end of this notebook, you will be able to:

- Build a character-level RNN that predicts the next character.
- Explain the roles of generator and discriminator networks in a GAN.
- Use GPT-2 and T5 style pre-trained models for generation tasks.
- Evaluate generated text with BLEU and ROUGE.


## 3.1 Text Generation and NLP

Text generation models predict the next character, word, subword, or sequence given earlier context.

| Application | Example |
|---|---|
| Chatbots | Continue a conversation. |
| Language translation | Generate a sentence in another language. |
| Technical writing | Draft structured explanations. |
| Text completion | Complete `"The cat is on the m"` as `"The cat is on the mat"`. |

RNNs, LSTMs, and GRUs remember past information to process sequential data.


## 3.2 Character-Level RNN

The PDF starts with a small RNN that learns to predict the next character. Character-level models are pedagogically useful because the vocabulary is tiny and every training example is easy to inspect.


In [ ]:
# PDF snippets: data setup and character indexes
import torch
import torch.nn as nn

torch.manual_seed(7)

data = "Hello how are you?"
chars = sorted(list(set(data)))
vocab_size = len(chars)

char_to_ix = {char: i for i, char in enumerate(chars)}
ix_to_char = {i: char for char, i in char_to_ix.items()}

print("Vocabulary:", chars)
print("Vocabulary size:", vocab_size)


In [ ]:
# PDF snippets: RNN model, forward propagation, and model creation
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(1, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out


model = RNNModel(input_size=vocab_size, hidden_size=16, output_size=vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [ ]:
# PDF snippet: preparing input and target data
inputs = [char_to_ix[ch] for ch in data[:-1]]
targets = [char_to_ix[ch] for ch in data[1:]]

inputs = torch.tensor(inputs, dtype=torch.long).view(1, -1)
inputs = nn.functional.one_hot(inputs, num_classes=vocab_size).float()
targets = torch.tensor([targets[-1]], dtype=torch.long)

print("Input shape: ", inputs.shape)
print("Target index:", targets.item(), "->", ix_to_char[targets.item()])


In [ ]:
# PDF snippet: training the RNN model
for epoch in range(100):
    model.train()
    outputs = model(inputs)
    loss = criterion(outputs, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1}/100, Loss: {loss.item():.4f}")


In [ ]:
# PDF snippet: testing the model
model.eval()
test_input = char_to_ix["H"]
test_input = nn.functional.one_hot(
    torch.tensor([[test_input]]),
    num_classes=vocab_size,
).float()

with torch.no_grad():
    predicted_output = model(test_input)

predicted_char_ix = torch.argmax(predicted_output, 1).item()
print(f"Test Input: H, Predicted Output: {ix_to_char[predicted_char_ix]}")


## 3.3 Generating Text from a Trained RNN

The next helper repeatedly feeds the last generated character back into the model. The sample is intentionally small, so the generated text is more of a mechanics demo than a high-quality language model.


In [ ]:
def generate_text(model, seed_char, length=50):
    model.eval()
    generated = seed_char
    current_char = seed_char

    for _ in range(length):
        idx = torch.tensor([[char_to_ix[current_char]]], dtype=torch.long)
        one_hot = nn.functional.one_hot(idx, num_classes=vocab_size).float()

        with torch.no_grad():
            output = model(one_hot)

        probabilities = torch.softmax(output, dim=-1)
        next_idx = torch.multinomial(probabilities, 1).item()
        next_char = ix_to_char[next_idx]
        generated += next_char
        current_char = next_char

    return generated


print(generate_text(model, "H", length=40))


## 3.4 Generative Adversarial Networks for Text Generation

The PDF introduces GANs as a way to generate new content that preserves statistical similarities with the training data.

| GAN component | Role |
|---|---|
| Generator | Creates fake samples from noise. |
| Discriminator | Distinguishes real samples from generated samples. |

The example below follows the PDF's simple tensor-based GAN structure. It represents synthetic text-like data as binary vectors so the mechanics remain executable.


In [ ]:
# PDF snippets: Generator, Discriminator, optimizers, and loss
seq_length = 5
data_tensor = torch.bernoulli(torch.full((20, seq_length), 0.5))


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(seq_length, seq_length),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(seq_length, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)


generator = Generator()
discriminator = Discriminator()

criterion = nn.BCELoss()
optimizer_gen = torch.optim.Adam(generator.parameters(), lr=0.001)
optimizer_disc = torch.optim.Adam(discriminator.parameters(), lr=0.001)


In [ ]:
# PDF snippets: training discriminator and generator
num_epochs = 50

for epoch in range(num_epochs):
    for real_data in data_tensor:
        real_data = real_data.unsqueeze(0)
        noise = torch.rand((1, seq_length))

        disc_real = discriminator(real_data)
        fake_data = generator(noise)
        disc_fake = discriminator(fake_data.detach())

        loss_disc = criterion(disc_real, torch.ones_like(disc_real)) + criterion(
            disc_fake, torch.zeros_like(disc_fake)
        )
        optimizer_disc.zero_grad()
        loss_disc.backward()
        optimizer_disc.step()

        disc_fake = discriminator(fake_data)
        loss_gen = criterion(disc_fake, torch.ones_like(disc_fake))
        optimizer_gen.zero_grad()
        loss_gen.backward()
        optimizer_gen.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}/{num_epochs}: "
            f"Generator loss: {loss_gen.item():.4f} "
            f"Discriminator loss: {loss_disc.item():.4f}"
        )


In [ ]:
# PDF snippet: printing real and generated data
print("\nReal data:")
print(data_tensor[:5])

print("\nGenerated data:")
for _ in range(5):
    noise = torch.rand((1, seq_length))
    generated_data = generator(noise)
    print(torch.round(generated_data).detach())


## 3.5 Pre-Trained Models for Text Generation

Pre-trained models are trained on extensive datasets and can perform well across text generation tasks such as sentiment-aware generation, text completion, and translation.

| Model | Use case | Limitation |
|---|---|---|
| GPT-2 / DistilGPT-2 | Text generation and completion | Large storage and compute requirements. |
| T5 / T5-small | Translation and summarization | Needs task-specific prompting. |

> **Note:** The next cells require `transformers` and model downloads. They are kept as complete PDF code snippets, with setup comments for environments that need them.


In [ ]:
# PDF snippets: GPT-2 tokenizer/model setup and generation
# Install if needed: pip install transformers
import torch

try:
    from transformers import GPT2Tokenizer, GPT2LMHeadModel

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    model_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")

    seed_text = "Once upon a time"
    input_ids = tokenizer.encode(seed_text, return_tensors="pt")
    output = model_gpt2.generate(
        input_ids,
        max_length=40,
        temperature=0.7,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    print("Generated Text:", generated_text)
except Exception as exc:
    print("GPT-2 example skipped:", exc)


In [ ]:
# PDF snippets: T5 language translation implementation and output decoding
try:
    from transformers import T5Tokenizer, T5ForConditionalGeneration

    tokenizer_t5 = T5Tokenizer.from_pretrained("t5-small")
    model_t5 = T5ForConditionalGeneration.from_pretrained("t5-small")

    input_prompt = "translate English to French: 'Hello, how are you?'"
    input_ids = tokenizer_t5.encode(input_prompt, return_tensors="pt")
    output = model_t5.generate(input_ids, max_length=100)

    generated_text = tokenizer_t5.decode(output[0], skip_special_tokens=True)
    print("Generated text:", generated_text)
except Exception as exc:
    print("T5 example skipped:", exc)


## 3.6 Evaluation Metrics for Text Generation

Standard classification metrics such as accuracy and F1 do not capture the quality of generated text. The PDF introduces BLEU and ROUGE.

| Metric | What it compares | Typical interpretation |
|---|---|---|
| BLEU | Generated text against references using n-gram overlap. | `1.0` is perfect overlap; `0` is no overlap. |
| ROUGE-N | Overlapping n-grams between generated and reference text. | Useful for summarization overlap. |
| ROUGE-L | Longest common subsequence. | Rewards sequence-level similarity. |

Limitations: these metrics evaluate word presence and overlap, not deep semantic understanding; they are sensitive to generated length and reference quality.


In [ ]:
# PDF snippet: calculating BLEU score with PyTorch
# Install if needed: pip install torchmetrics
generated_text = ["the cat is on the mat"]
real_text = [["there is a cat on the mat", "a cat is on the mat"]]

try:
    from torchmetrics.text import BLEUScore

    bleu = BLEUScore()
    bleu_metric = bleu(generated_text, real_text)
    print("BLEU Score:", bleu_metric.item())
except Exception as exc:
    print("BLEU example skipped:", exc)


In [ ]:
# PDF snippet: calculating ROUGE score with PyTorch
generated_text = "Hello, how are you doing?"
real_text = "Hello, how are you?"

try:
    from torchmetrics.text import ROUGEScore

    rouge = ROUGEScore()
    rouge_score = rouge([generated_text], [[real_text]])
    print("ROUGE Score:", rouge_score)
except Exception as exc:
    print("ROUGE example skipped:", exc)


## Chapter Summary

| Area | What you learned |
|---|---|
| RNN generation | Predict the next character using recurrent state. |
| GANs | Train a generator and discriminator in opposition. |
| GPT-2 | Use a pre-trained autoregressive model for continuation. |
| T5 | Use text-to-text prompting for translation. |
| BLEU/ROUGE | Evaluate n-gram and sequence overlap with reference text. |
